Explore data structure: tissues, mice, age groups

In [1]:

import pandas as pd

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nAge groups: {df['age'].unique()}")
print(f"\nMice: {sorted(df['mouse'].unique())}")
print(f"\nTissues (tissue_ct): {sorted(df['tissue_ct'].unique())}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Extract tissue (before the dash)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
print(f"\nTissues (broad): {sorted(df['tissue'].unique())}")

# Per tissue, per mouse total cell count
print(f"\nMice per age:")
for age in sorted(df['age'].unique()):
    mice = sorted(df[df['age'] == age]['mouse'].unique())
    print(f"  {age}: {mice}")


Shape: (807, 7)

Columns: ['mouse', 'proj', 'age', 'RNA_reads_per_cell', 'total_reads', 'cell_count', 'tissue_ct']

Age groups: ['age27M' 'age3M']

Mice: ['mouse122', 'mouse123', 'mouse124', 'mouse125', 'mouse126', 'mouse127', 'mouse128', 'mouse129', 'mouse130', 'mouse131', 'mouse135', 'mouse136', 'mouse137', 'mouse138', 'mouse139', 'mouse140', 'mouse141', 'mouse142', 'mouse143']

Tissues (tissue_ct): ['BAT-ASPC', 'BAT-Adipocyte', 'BAT-B_cell', 'BAT-Brown_adipocyte', 'BAT-MSC', 'BAT-Mural_cell', 'BAT-Myeloid_cell', 'BAT-Non_MSC', 'BAT-Skeletal_muscle_cell', 'BAT-T_cell', 'BAT-Vascular_endothelial_cell', 'brainCB-Asc_Glia', 'brainCB-BG_Glia', 'brainCB-Choroid_Other', 'brainCB-Doublet', 'brainCB-Endo_Other', 'brainCB-Ependymal_Other', 'brainCB-Fibroblasts_Other', 'brainCB-GC_ExN', 'brainCB-Golgi_InN', 'brainCB-MLI1_InN', 'brainCB-MLI2_InN', 'brainCB-Mgc_Glia', 'brainCB-Ogc_Glia', 'brainCB-Opc_Glia', 'brainCB-PC_InN', 'brainCB-PLI_InN', 'brainCB-Peri_Other', 'brainCB-SMC_Other', 'brainCB-

Plot total cell counts per tissue, colored by mouse with age-based color palette

In [3]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

os.makedirs("/mnt/results", exist_ok=True)

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]

# === Metadata ===
df_meta = df[['mouse', 'age']].drop_duplicates()

# === Color map ===
n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],   # skip very light shades
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}

mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

# === Column order: 3M first, 27M second ===
mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M

# === Total cell count per tissue per mouse ===
total_counts = (
    df.groupby(["tissue", "mouse"])["cell_count"]
    .sum()
    .reset_index()
)

# Tissue order (alphabetical, or customize)
tissue_order = sorted(total_counts["tissue"].unique())

# === Plot ===
sns.set_theme(style="ticks", font_scale=1.0)

n_tissues = len(tissue_order)
fig, axes = plt.subplots(
    1, n_tissues,
    figsize=(2.2 * n_tissues, 5),
    sharey=False
)

for ax, tissue in zip(axes, tissue_order):
    sub = total_counts[total_counts["tissue"] == tissue].copy()
    # Keep ordered_mice order, only those present
    sub_mice = [m for m in ordered_mice if m in sub["mouse"].values]
    sub = sub.set_index("mouse").reindex(sub_mice).reset_index()

    colors = [mouse_colors[m] for m in sub["mouse"]]
    bars = ax.bar(range(len(sub)), sub["cell_count"], color=colors, width=0.7, edgecolor="white", linewidth=0.4)

    ax.set_title(tissue, fontsize=9, fontweight="bold", pad=4)
    ax.set_xticks([])
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k" if x >= 1000 else str(int(x))))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=7)

axes[0].set_ylabel("Total cell count", fontsize=9)

# === Legend ===
legend_handles = []
# 3M group label
legend_handles.append(mpatches.Patch(color="none", label="── age3M ──"))
for m in mice_3M:
    legend_handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
legend_handles.append(mpatches.Patch(color="none", label=""))
legend_handles.append(mpatches.Patch(color="none", label="── age27M ──"))
for m in mice_27M:
    legend_handles.append(mpatches.Patch(color=mouse_colors[m], label=m))

fig.legend(
    handles=legend_handles,
    loc="center right",
    bbox_to_anchor=(1.08, 0.5),
    fontsize=7.5,
    frameon=False,
    handlelength=1.2,
    handleheight=1.0,
)

fig.suptitle("Total cell counts per tissue", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()

plt.savefig("/mnt/results/cell_counts_qc_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_counts_qc_per_tissue.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")


Saved.


Replot with larger fonts and labels

In [5]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]

df_meta = df[['mouse', 'age']].drop_duplicates()

n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}

mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M

total_counts = (
    df.groupby(["tissue", "mouse"])["cell_count"]
    .sum()
    .reset_index()
)

tissue_order = sorted(total_counts["tissue"].unique())

# === Plot with larger fonts ===
sns.set_theme(style="ticks", font_scale=1.0)

n_tissues = len(tissue_order)
fig, axes = plt.subplots(
    1, n_tissues,
    figsize=(3.0 * n_tissues, 6.5),
    sharey=False
)

for ax, tissue in zip(axes, tissue_order):
    sub = total_counts[total_counts["tissue"] == tissue].copy()
    sub_mice = [m for m in ordered_mice if m in sub["mouse"].values]
    sub = sub.set_index("mouse").reindex(sub_mice).reset_index()

    colors = [mouse_colors[m] for m in sub["mouse"]]
    ax.bar(range(len(sub)), sub["cell_count"], color=colors, width=0.7, edgecolor="white", linewidth=0.5)

    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=6)
    ax.set_xticks([])
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=11)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4, integer=True))

axes[0].set_ylabel("Total cell count", fontsize=13)

# === Legend ===
legend_handles = []
legend_handles.append(mpatches.Patch(color="none", label="age3M"))
for m in mice_3M:
    legend_handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
legend_handles.append(mpatches.Patch(color="none", label=""))
legend_handles.append(mpatches.Patch(color="none", label="age27M"))
for m in mice_27M:
    legend_handles.append(mpatches.Patch(color=mouse_colors[m], label=m))

fig.legend(
    handles=legend_handles,
    loc="center right",
    bbox_to_anchor=(1.07, 0.5),
    fontsize=11,
    frameon=False,
    handlelength=1.4,
    handleheight=1.2,
    title_fontsize=12,
)

fig.suptitle("Total cell counts per tissue", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

plt.savefig("/mnt/results/cell_counts_qc_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_counts_qc_per_tissue.png", dpi=150, bbox_inches="tight")
plt.show()
print("Done.")


Done.


Replot with 2 rows of tissue panels

In [7]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]

df_meta = df[['mouse', 'age']].drop_duplicates()

n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}

mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M

total_counts = (
    df.groupby(["tissue", "mouse"])["cell_count"]
    .sum()
    .reset_index()
)

tissue_order = sorted(total_counts["tissue"].unique())  # 12 tissues

# === 2 rows x 6 cols ===
n_cols = 6
n_rows = 2
sns.set_theme(style="ticks", font_scale=1.0)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(3.2 * n_cols, 6.0 * n_rows),
    sharey=False
)
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = total_counts[total_counts["tissue"] == tissue].copy()
    sub_mice = [m for m in ordered_mice if m in sub["mouse"].values]
    sub = sub.set_index("mouse").reindex(sub_mice).reset_index()

    colors = [mouse_colors[m] for m in sub["mouse"]]
    ax.bar(range(len(sub)), sub["cell_count"], color=colors, width=0.7, edgecolor="white", linewidth=0.5)

    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=6)
    ax.set_xticks([])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=11)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4, integer=True))

# y-axis label only on leftmost panels
for row in range(n_rows):
    axes[row, 0].set_ylabel("Total cell count", fontsize=13)

# Hide unused axes (if any)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

# === Legend ===
legend_handles = []
legend_handles.append(mpatches.Patch(color="none", label="age3M"))
for m in mice_3M:
    legend_handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
legend_handles.append(mpatches.Patch(color="none", label=""))
legend_handles.append(mpatches.Patch(color="none", label="age27M"))
for m in mice_27M:
    legend_handles.append(mpatches.Patch(color=mouse_colors[m], label=m))

fig.legend(
    handles=legend_handles,
    loc="center right",
    bbox_to_anchor=(1.06, 0.5),
    fontsize=11,
    frameon=False,
    handlelength=1.4,
    handleheight=1.2,
)

fig.suptitle("Total cell counts per tissue", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()

plt.savefig("/mnt/results/cell_counts_qc_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_counts_qc_per_tissue.png", dpi=150, bbox_inches="tight")
plt.show()
print("Done.")


Done.


Setup: load data and define shared color utilities

In [9]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

os.makedirs("/mnt/results/cell_type_qc", exist_ok=True)

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

df_meta = df[['mouse', 'age']].drop_duplicates()

n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}

mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M

tissue_order = sorted(df['tissue'].unique())
print("Tissues:", tissue_order)
print("Mice 3M:", mice_3M)
print("Mice 27M:", mice_27M)

# Mouse legend handles (reused across plots)
def make_mouse_legend():
    handles = []
    handles.append(mpatches.Patch(color="none", label="age3M"))
    for m in mice_3M:
        handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
    handles.append(mpatches.Patch(color="none", label=""))
    handles.append(mpatches.Patch(color="none", label="age27M"))
    for m in mice_27M:
        handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
    return handles


Tissues: ['BAT', 'brainCB', 'brainFC', 'brainHip', 'colon', 'heart', 'kidney', 'liver', 'lung', 'muscle', 'spleen', 'testis']
Mice 3M: ['mouse135', 'mouse136', 'mouse137', 'mouse138', 'mouse139', 'mouse140', 'mouse141', 'mouse142', 'mouse143']
Mice 27M: ['mouse122', 'mouse123', 'mouse124', 'mouse125', 'mouse126', 'mouse127', 'mouse128', 'mouse129', 'mouse130', 'mouse131']


Stacked bar — one plot per tissue (12 figures)

In [11]:

# Generate a distinct color palette for cell types (per tissue)
from matplotlib.cm import get_cmap

sns.set_theme(style="ticks", font_scale=1.0)

for tissue in tissue_order:
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    n_ct = len(cell_types)

    # Cell type colors: tab20 / tab20b extended
    ct_palette = (sns.color_palette("tab20", 20) + sns.color_palette("tab20b", 20))[:n_ct]
    ct_colors = {ct: ct_palette[i] for i, ct in enumerate(cell_types)}

    # Pivot: rows=mouse, cols=cell_type
    pivot = sub.pivot_table(index='mouse', columns='cell_type', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex([m for m in ordered_mice if m in pivot.index])

    fig, ax = plt.subplots(figsize=(max(8, len(pivot)*0.9 + 2), 6))

    bottom = np.zeros(len(pivot))
    for ct in cell_types:
        vals = pivot[ct].values if ct in pivot.columns else np.zeros(len(pivot))
        bars = ax.bar(range(len(pivot)), vals, bottom=bottom,
                      color=ct_colors[ct], label=ct, width=0.7, edgecolor='white', linewidth=0.4)
        bottom += vals

    # X tick labels colored by mouse
    ax.set_xticks(range(len(pivot)))
    ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=11)
    for tick, mouse in zip(ax.get_xticklabels(), pivot.index):
        tick.set_color(mouse_colors.get(mouse, 'black'))
        tick.set_fontweight('bold')

    ax.set_ylabel("Cell count", fontsize=13)
    ax.set_title(f"{tissue} — cell type composition", fontsize=14, fontweight='bold')
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis='y', labelsize=11)

    # Cell type legend (right side)
    ct_handles = [mpatches.Patch(color=ct_colors[ct], label=ct) for ct in cell_types]
    ax.legend(handles=ct_handles, loc='upper left', bbox_to_anchor=(1.01, 1),
              fontsize=9, frameon=False, title='Cell type', title_fontsize=10)

    plt.tight_layout()
    plt.savefig(f"/mnt/results/cell_type_qc/stacked_bar_{tissue}.svg", bbox_inches='tight')
    plt.savefig(f"/mnt/results/cell_type_qc/stacked_bar_{tissue}.png", dpi=150, bbox_inches='tight')
    plt.close()

print("Done: stacked bar per tissue")


Done: stacked bar per tissue


Stacked bar — all tissues combined in one figure (2 rows x 6 cols)

In [13]:

sns.set_theme(style="ticks", font_scale=1.0)

# Build a global cell-type color map per tissue (each tissue has its own palette)
tissue_ct_colors = {}
for tissue in tissue_order:
    sub = df[df['tissue'] == tissue]
    cell_types = sorted(sub['cell_type'].unique())
    n_ct = len(cell_types)
    ct_palette = (sns.color_palette("tab20", 20) + sns.color_palette("tab20b", 20))[:n_ct]
    tissue_ct_colors[tissue] = {ct: ct_palette[i] for i, ct in enumerate(cell_types)}

n_cols = 6
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    ct_colors = tissue_ct_colors[tissue]

    pivot = sub.pivot_table(index='mouse', columns='cell_type', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex([m for m in ordered_mice if m in pivot.index])

    bottom = np.zeros(len(pivot))
    for ct in cell_types:
        vals = pivot[ct].values if ct in pivot.columns else np.zeros(len(pivot))
        ax.bar(range(len(pivot)), vals, bottom=bottom,
               color=ct_colors[ct], label=ct, width=0.7, edgecolor='white', linewidth=0.3)
        bottom += vals

    ax.set_xticks(range(len(pivot)))
    ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=9)
    for tick, mouse in zip(ax.get_xticklabels(), pivot.index):
        tick.set_color(mouse_colors.get(mouse, 'black'))
        tick.set_fontweight('bold')

    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))

    # Per-panel legend (compact)
    ct_handles = [mpatches.Patch(color=ct_colors[ct], label=ct) for ct in cell_types]
    ax.legend(handles=ct_handles, fontsize=6.5, frameon=False,
              loc='upper left', bbox_to_anchor=(1.01, 1), title='Cell type', title_fontsize=7.5)

# y-axis labels on leftmost panels
for row in range(n_rows):
    axes[row, 0].set_ylabel("Cell count", fontsize=12)

# Hide unused axes
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Cell type composition per tissue", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/stacked_bar_all_tissues.svg", bbox_inches='tight')
plt.savefig("/mnt/results/cell_type_qc/stacked_bar_all_tissues.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done: stacked bar all tissues combined")


Done: stacked bar all tissues combined


Heatmap — one plot per tissue (12 figures)

In [15]:

sns.set_theme(style="ticks", font_scale=1.0)

for tissue in tissue_order:
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())

    pivot = sub.pivot_table(index='cell_type', columns='mouse', values='cell_count', aggfunc='sum').fillna(0)
    # reorder columns by ordered_mice
    pivot = pivot.reindex(columns=[m for m in ordered_mice if m in pivot.columns])

    n_ct = len(cell_types)
    n_mice = pivot.shape[1]
    fig_h = max(4, n_ct * 0.45 + 1.5)
    fig_w = max(6, n_mice * 0.7 + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')

    # Axes
    ax.set_xticks(range(n_mice))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=11, fontweight='bold')
    for tick, mouse in zip(ax.get_xticklabels(), pivot.columns):
        tick.set_color(mouse_colors.get(mouse, 'black'))

    ax.set_yticks(range(n_ct))
    ax.set_yticklabels(pivot.index, fontsize=10)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label("Cell count", fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    cbar.formatter = plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x)))
    cbar.update_ticks()

    # Age separator line between 3M and 27M columns
    n_3m_present = sum(1 for m in mice_3M if m in pivot.columns)
    if 0 < n_3m_present < n_mice:
        ax.axvline(x=n_3m_present - 0.5, color='black', linewidth=1.5, linestyle='--', alpha=0.6)
        ax.text(n_3m_present/2 - 0.5, -1.2, 'age3M', ha='center', va='top', fontsize=10, color='steelblue', fontweight='bold')
        ax.text(n_3m_present + (n_mice - n_3m_present)/2 - 0.5, -1.2, 'age27M', ha='center', va='top', fontsize=10, color='firebrick', fontweight='bold')

    ax.set_title(f"{tissue} — cell count heatmap", fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel("")
    ax.set_ylabel("Cell type", fontsize=12)

    plt.tight_layout()
    plt.savefig(f"/mnt/results/cell_type_qc/heatmap_{tissue}.svg", bbox_inches='tight')
    plt.savefig(f"/mnt/results/cell_type_qc/heatmap_{tissue}.png", dpi=150, bbox_inches='tight')
    plt.close()

print("Done: heatmap per tissue")


Done: heatmap per tissue


Fix heatmap — move age labels below x-axis to avoid title overlap

In [17]:

sns.set_theme(style="ticks", font_scale=1.0)

for tissue in tissue_order:
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())

    pivot = sub.pivot_table(index='cell_type', columns='mouse', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex(columns=[m for m in ordered_mice if m in pivot.columns])

    n_ct = len(cell_types)
    n_mice = pivot.shape[1]
    fig_h = max(4, n_ct * 0.45 + 2.0)
    fig_w = max(6, n_mice * 0.7 + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')

    # X axis — mouse labels colored by age
    ax.set_xticks(range(n_mice))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=11, fontweight='bold')
    for tick, mouse in zip(ax.get_xticklabels(), pivot.columns):
        tick.set_color(mouse_colors.get(mouse, 'black'))

    # Y axis — cell types
    ax.set_yticks(range(n_ct))
    ax.set_yticklabels(pivot.index, fontsize=10)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label("Cell count", fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    cbar.formatter = plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x)))
    cbar.update_ticks()

    # Age separator + bracket-style labels below x-axis
    n_3m_present = sum(1 for m in mice_3M if m in pivot.columns)
    if 0 < n_3m_present < n_mice:
        ax.axvline(x=n_3m_present - 0.5, color='black', linewidth=1.5, linestyle='--', alpha=0.6)

    ax.set_title(f"{tissue} — cell count heatmap", fontsize=14, fontweight='bold', pad=8)
    ax.set_ylabel("Cell type", fontsize=12)

    plt.tight_layout()

    # Add age group labels below x-axis after layout is fixed
    if 0 < n_3m_present < n_mice:
        fig.canvas.draw()
        # Use figure-level annotation in axis coordinates
        ax.annotate('age3M', xy=((n_3m_present/2 - 0.5) / (n_mice - 1), -0.28),
                    xycoords='axes fraction', ha='center', fontsize=10,
                    color='steelblue', fontweight='bold')
        ax.annotate('age27M', xy=((n_3m_present + (n_mice - n_3m_present)/2 - 0.5) / (n_mice - 1), -0.28),
                    xycoords='axes fraction', ha='center', fontsize=10,
                    color='firebrick', fontweight='bold')

    plt.savefig(f"/mnt/results/cell_type_qc/heatmap_{tissue}.svg", bbox_inches='tight')
    plt.savefig(f"/mnt/results/cell_type_qc/heatmap_{tissue}.png", dpi=150, bbox_inches='tight')
    plt.close()

print("Done: heatmap per tissue (fixed)")


Done: heatmap per tissue (fixed)


Heatmap — all tissues combined, subplots 2x6

In [19]:

sns.set_theme(style="ticks", font_scale=1.0)

n_cols = 6
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 7.0 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()

    pivot = sub.pivot_table(index='cell_type', columns='mouse', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex(columns=[m for m in ordered_mice if m in pivot.columns])

    n_ct = pivot.shape[0]
    n_mice = pivot.shape[1]

    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')

    # X axis
    ax.set_xticks(range(n_mice))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8.5, fontweight='bold')
    for tick, mouse in zip(ax.get_xticklabels(), pivot.columns):
        tick.set_color(mouse_colors.get(mouse, 'black'))

    # Y axis
    ax.set_yticks(range(n_ct))
    ax.set_yticklabels(pivot.index, fontsize=8)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    cbar.ax.tick_params(labelsize=7)
    cbar.formatter = plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x)))
    cbar.update_ticks()

    # Age separator
    n_3m_present = sum(1 for m in mice_3M if m in pivot.columns)
    if 0 < n_3m_present < n_mice:
        ax.axvline(x=n_3m_present - 0.5, color='black', linewidth=1.2, linestyle='--', alpha=0.6)

    ax.set_title(tissue, fontsize=12, fontweight='bold', pad=5)

# Hide unused
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

# Shared age group labels as figure-level text
fig.text(0.15, 0.01, 'age3M', ha='center', fontsize=11, color='steelblue', fontweight='bold')
fig.text(0.35, 0.01, 'age27M', ha='center', fontsize=11, color='firebrick', fontweight='bold')
fig.text(0.01, 0.5, 'Cell type', va='center', rotation='vertical', fontsize=12)

fig.suptitle("Cell count heatmap per tissue", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=2)

plt.savefig("/mnt/results/cell_type_qc/heatmap_all_tissues.svg", bbox_inches='tight')
plt.savefig("/mnt/results/cell_type_qc/heatmap_all_tissues.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done: heatmap all tissues combined")


Done: heatmap all tissues combined


Grouped bar — one plot per tissue (12 figures)

In [21]:

sns.set_theme(style="ticks", font_scale=1.0)

for tissue in tissue_order:
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    n_ct = len(cell_types)

    pivot = sub.pivot_table(index='cell_type', columns='mouse', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex(columns=[m for m in ordered_mice if m in pivot.columns])
    mice_present = list(pivot.columns)
    n_mice = len(mice_present)

    bar_width = 0.8 / n_mice
    x = np.arange(n_ct)

    fig_w = max(10, n_ct * (bar_width * n_mice + 0.3) + 3)
    fig_h = 5.5

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    for i, mouse in enumerate(mice_present):
        offset = (i - n_mice / 2 + 0.5) * bar_width
        vals = pivot.loc[:, mouse].reindex(cell_types).fillna(0).values
        ax.bar(x + offset, vals, width=bar_width * 0.92,
               color=mouse_colors[mouse], label=mouse, edgecolor='white', linewidth=0.3)

    ax.set_xticks(x)
    ax.set_xticklabels(cell_types, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel("Cell count", fontsize=12)
    ax.set_title(f"{tissue} — cell counts by cell type", fontsize=14, fontweight='bold')
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))

    # Mouse legend
    ax.legend(handles=make_mouse_legend(), loc='upper left', bbox_to_anchor=(1.01, 1),
              fontsize=9, frameon=False)

    plt.tight_layout()
    plt.savefig(f"/mnt/results/cell_type_qc/grouped_bar_{tissue}.svg", bbox_inches='tight')
    plt.savefig(f"/mnt/results/cell_type_qc/grouped_bar_{tissue}.png", dpi=150, bbox_inches='tight')
    plt.close()

print("Done: grouped bar per tissue")


Done: grouped bar per tissue


Grouped bar — all tissues combined, 2x6 subplots

In [23]:

sns.set_theme(style="ticks", font_scale=1.0)

n_cols = 6
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    n_ct = len(cell_types)

    pivot = sub.pivot_table(index='cell_type', columns='mouse', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex(columns=[m for m in ordered_mice if m in pivot.columns])
    mice_present = list(pivot.columns)
    n_mice = len(mice_present)

    bar_width = 0.8 / n_mice
    x = np.arange(n_ct)

    for i, mouse in enumerate(mice_present):
        offset = (i - n_mice / 2 + 0.5) * bar_width
        vals = pivot.loc[:, mouse].reindex(cell_types).fillna(0).values
        ax.bar(x + offset, vals, width=bar_width * 0.92,
               color=mouse_colors[mouse], label=mouse, edgecolor='white', linewidth=0.2)

    ax.set_xticks(x)
    ax.set_xticklabels(cell_types, rotation=45, ha='right', fontsize=7.5)
    ax.set_title(tissue, fontsize=12, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis='y', labelsize=9)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))

# y-axis labels on leftmost panels
for row in range(n_rows):
    axes[row, 0].set_ylabel("Cell count", fontsize=11)

# Hide unused axes
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

# Shared mouse legend on the right
legend_handles = make_mouse_legend()
fig.legend(handles=legend_handles, loc='center right', bbox_to_anchor=(1.06, 0.5),
           fontsize=9.5, frameon=False, handlelength=1.4, handleheight=1.2)

fig.suptitle("Cell counts by cell type per tissue", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=5, w_pad=2)

plt.savefig("/mnt/results/cell_type_qc/grouped_bar_all_tissues.svg", bbox_inches='tight')
plt.savefig("/mnt/results/cell_type_qc/grouped_bar_all_tissues.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done: grouped bar all tissues combined")


Done: grouped bar all tissues combined


Stacked bar all tissues — normalized to 100% per mouse

In [25]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

df_meta = df[['mouse', 'age']].drop_duplicates()

n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}

mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M

tissue_order = sorted(df['tissue'].unique())

# ── Build tissue-level cell type color maps ──────────────────────────────────
tissue_ct_colors = {}
for tissue in tissue_order:
    sub = df[df['tissue'] == tissue]
    cell_types = sorted(sub['cell_type'].unique())
    n_ct = len(cell_types)
    ct_palette = (sns.color_palette("tab20", 20) + sns.color_palette("tab20b", 20))[:n_ct]
    tissue_ct_colors[tissue] = {ct: ct_palette[i] for i, ct in enumerate(cell_types)}

# ── Plot ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="ticks", font_scale=1.0)

n_cols = 6
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    ct_colors = tissue_ct_colors[tissue]

    pivot = sub.pivot_table(index='mouse', columns='cell_type', values='cell_count', aggfunc='sum').fillna(0)
    pivot = pivot.reindex([m for m in ordered_mice if m in pivot.index])

    # Normalize each row (mouse) to 100%
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    bottom = np.zeros(len(pivot_pct))
    for ct in cell_types:
        vals = pivot_pct[ct].values if ct in pivot_pct.columns else np.zeros(len(pivot_pct))
        ax.bar(range(len(pivot_pct)), vals, bottom=bottom,
               color=ct_colors[ct], label=ct, width=0.7, edgecolor='white', linewidth=0.3)
        bottom += vals

    # X tick labels colored by mouse
    ax.set_xticks(range(len(pivot_pct)))
    ax.set_xticklabels(pivot_pct.index, rotation=45, ha='right', fontsize=9, fontweight='bold')
    for tick, mouse in zip(ax.get_xticklabels(), pivot_pct.index):
        tick.set_color(mouse_colors.get(mouse, 'black'))

    ax.set_ylim(0, 100)
    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.set_major_locator(plt.MultipleLocator(25))

    # Per-panel cell type legend
    ct_handles = [mpatches.Patch(color=ct_colors[ct], label=ct) for ct in cell_types]
    ax.legend(handles=ct_handles, fontsize=6.5, frameon=False,
              loc='upper left', bbox_to_anchor=(1.01, 1), title='Cell type', title_fontsize=7.5)

# y-axis labels on leftmost panels
for row in range(n_rows):
    axes[row, 0].set_ylabel("Cell type proportion (%)", fontsize=12)

# Hide unused axes
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Cell type composition per tissue (% of total cells)", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/stacked_bar_pct_all_tissues.svg", bbox_inches='tight')
plt.savefig("/mnt/results/cell_type_qc/stacked_bar_pct_all_tissues.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done.")


Done.


Generate 3 total_reads QC plots: per-tissue bar, stacked bar, and 100% stacked bar

In [27]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

os.makedirs("/mnt/results/cell_type_qc", exist_ok=True)

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

df_meta = df[['mouse', 'age']].drop_duplicates()

n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}

mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M

tissue_order = sorted(df['tissue'].unique())

def make_mouse_legend():
    handles = []
    handles.append(mpatches.Patch(color="none", label="age3M"))
    for m in mice_3M:
        handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
    handles.append(mpatches.Patch(color="none", label=""))
    handles.append(mpatches.Patch(color="none", label="age27M"))
    for m in mice_27M:
        handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
    return handles

# ── Cell type color maps ──────────────────────────────────────────────────────
tissue_ct_colors = {}
for tissue in tissue_order:
    sub = df[df['tissue'] == tissue]
    cell_types = sorted(sub['cell_type'].unique())
    n_ct = len(cell_types)
    ct_palette = (sns.color_palette("tab20", 20) + sns.color_palette("tab20b", 20))[:n_ct]
    tissue_ct_colors[tissue] = {ct: ct_palette[i] for i, ct in enumerate(cell_types)}

sns.set_theme(style="ticks", font_scale=1.0)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. cell_counts_qc_per_tissue — total_reads, 2 rows x 6 cols
# ═══════════════════════════════════════════════════════════════════════════════
total_reads = (
    df.groupby(["tissue", "mouse"])["total_reads"]
    .sum()
    .reset_index()
)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 6.0 * n_rows), sharey=False)
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = total_reads[total_reads["tissue"] == tissue].copy()
    sub_mice = [m for m in ordered_mice if m in sub["mouse"].values]
    sub = sub.set_index("mouse").reindex(sub_mice).reset_index()

    colors = [mouse_colors[m] for m in sub["mouse"]]
    ax.bar(range(len(sub)), sub["total_reads"], color=colors, width=0.7, edgecolor="white", linewidth=0.5)

    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=6)
    ax.set_xticks([])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{x/1000:.0f}k"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=11)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4, integer=True))

for row in range(n_rows):
    axes[row, 0].set_ylabel("Total reads", fontsize=13)

for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.legend(handles=make_mouse_legend(), loc="center right", bbox_to_anchor=(1.07, 0.5),
           fontsize=11, frameon=False, handlelength=1.4, handleheight=1.2)
fig.suptitle("Total reads per tissue", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/mnt/results/cell_type_qc/total_reads_qc_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/total_reads_qc_per_tissue.png", dpi=150, bbox_inches="tight")
plt.close()
print("1/3 done")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. stacked_bar_all_tissues — total_reads by cell type (absolute)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    ct_colors = tissue_ct_colors[tissue]

    pivot = sub.pivot_table(index='mouse', columns='cell_type', values='total_reads', aggfunc='sum').fillna(0)
    pivot = pivot.reindex([m for m in ordered_mice if m in pivot.index])

    bottom = np.zeros(len(pivot))
    for ct in cell_types:
        vals = pivot[ct].values if ct in pivot.columns else np.zeros(len(pivot))
        ax.bar(range(len(pivot)), vals, bottom=bottom,
               color=ct_colors[ct], label=ct, width=0.7, edgecolor='white', linewidth=0.3)
        bottom += vals

    ax.set_xticks(range(len(pivot)))
    ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=9, fontweight='bold')
    for tick, mouse in zip(ax.get_xticklabels(), pivot.index):
        tick.set_color(mouse_colors.get(mouse, 'black'))

    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{x/1000:.0f}k"))
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))

    ct_handles = [mpatches.Patch(color=ct_colors[ct], label=ct) for ct in cell_types]
    ax.legend(handles=ct_handles, fontsize=6.5, frameon=False,
              loc='upper left', bbox_to_anchor=(1.01, 1), title='Cell type', title_fontsize=7.5)

for row in range(n_rows):
    axes[row, 0].set_ylabel("Total reads", fontsize=12)

for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Total reads by cell type per tissue", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)
plt.savefig("/mnt/results/cell_type_qc/stacked_bar_reads_all_tissues.svg", bbox_inches='tight')
plt.savefig("/mnt/results/cell_type_qc/stacked_bar_reads_all_tissues.png", dpi=150, bbox_inches='tight')
plt.close()
print("2/3 done")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. stacked_bar_pct_all_tissues — total_reads normalized to 100%
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()
    cell_types = sorted(sub['cell_type'].unique())
    ct_colors = tissue_ct_colors[tissue]

    pivot = sub.pivot_table(index='mouse', columns='cell_type', values='total_reads', aggfunc='sum').fillna(0)
    pivot = pivot.reindex([m for m in ordered_mice if m in pivot.index])
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    bottom = np.zeros(len(pivot_pct))
    for ct in cell_types:
        vals = pivot_pct[ct].values if ct in pivot_pct.columns else np.zeros(len(pivot_pct))
        ax.bar(range(len(pivot_pct)), vals, bottom=bottom,
               color=ct_colors[ct], label=ct, width=0.7, edgecolor='white', linewidth=0.3)
        bottom += vals

    ax.set_xticks(range(len(pivot_pct)))
    ax.set_xticklabels(pivot_pct.index, rotation=45, ha='right', fontsize=9, fontweight='bold')
    for tick, mouse in zip(ax.get_xticklabels(), pivot_pct.index):
        tick.set_color(mouse_colors.get(mouse, 'black'))

    ax.set_ylim(0, 100)
    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
    ax.tick_params(axis='y', labelsize=10)
    ax.yaxis.set_major_locator(plt.MultipleLocator(25))

    ct_handles = [mpatches.Patch(color=ct_colors[ct], label=ct) for ct in cell_types]
    ax.legend(handles=ct_handles, fontsize=6.5, frameon=False,
              loc='upper left', bbox_to_anchor=(1.01, 1), title='Cell type', title_fontsize=7.5)

for row in range(n_rows):
    axes[row, 0].set_ylabel("Total reads proportion (%)", fontsize=12)

for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Total reads by cell type per tissue (% of total reads)", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)
plt.savefig("/mnt/results/cell_type_qc/stacked_bar_reads_pct_all_tissues.svg", bbox_inches='tight')
plt.savefig("/mnt/results/cell_type_qc/stacked_bar_reads_pct_all_tissues.png", dpi=150, bbox_inches='tight')
plt.close()
print("3/3 done")


1/3 done
2/3 done
3/3 done


Horizontal barplot: total sequencing depth per tissue (3M + 27M combined)

In [29]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]

# Total reads per tissue per age group, then sum across mice
depth = (
    df.groupby(["tissue", "age"])["total_reads"]
    .sum()
    .reset_index()
)

# Pivot: rows=tissue, cols=age
pivot = depth.pivot(index="tissue", columns="age", values="total_reads").fillna(0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=True)  # ascending for horizontal bar

tissues = pivot.index.tolist()
age3M_vals  = pivot["age3M"].values
age27M_vals = pivot["age27M"].values

sns.set_theme(style="ticks", font_scale=1.0)
fig, ax = plt.subplots(figsize=(8, 6))

y = np.arange(len(tissues))
bar_h = 0.55

# Stacked: 3M first, then 27M on top
b1 = ax.barh(y, age3M_vals,  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.4)
b2 = ax.barh(y, age27M_vals, height=bar_h, left=age3M_vals, color="#D6604D", label="age27M", edgecolor="white", linewidth=0.4)

# Total label at end of each bar
for i, (v3, v27) in enumerate(zip(age3M_vals, age27M_vals)):
    total = v3 + v27
    ax.text(total + total * 0.01, i, f"{total/1e6:.0f}M",
            va="center", ha="left", fontsize=10)

ax.set_yticks(y)
ax.set_yticklabels(tissues, fontsize=12)
ax.set_xlabel("Total reads", fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.tick_params(axis="x", labelsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Total sequencing depth per tissue\n(age3M + age27M)", fontsize=14, fontweight="bold", pad=10)
ax.legend(fontsize=11, frameon=False, loc="lower right")

# Add some right margin for labels
ax.set_xlim(0, pivot["total"].max() * 1.15)

plt.tight_layout()
plt.savefig("/mnt/results/cell_type_qc/total_depth_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/total_depth_per_tissue.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


Done.


Horizontal barplot: total cell counts per tissue (3M + 27M combined)

In [31]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]

depth = (
    df.groupby(["tissue", "age"])["cell_count"]
    .sum()
    .reset_index()
)

pivot = depth.pivot(index="tissue", columns="age", values="cell_count").fillna(0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=True)

tissues = pivot.index.tolist()
age3M_vals  = pivot["age3M"].values
age27M_vals = pivot["age27M"].values

sns.set_theme(style="ticks", font_scale=1.0)
fig, ax = plt.subplots(figsize=(8, 6))

y = np.arange(len(tissues))
bar_h = 0.55

ax.barh(y, age3M_vals,  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.4)
ax.barh(y, age27M_vals, height=bar_h, left=age3M_vals, color="#D6604D", label="age27M", edgecolor="white", linewidth=0.4)

for i, (v3, v27) in enumerate(zip(age3M_vals, age27M_vals)):
    total = v3 + v27
    ax.text(total + pivot["total"].max() * 0.01, i, f"{total/1000:.0f}k",
            va="center", ha="left", fontsize=10)

ax.set_yticks(y)
ax.set_yticklabels(tissues, fontsize=12)
ax.set_xlabel("Total cell count", fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.tick_params(axis="x", labelsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Total cell counts per tissue\n(age3M + age27M)", fontsize=14, fontweight="bold", pad=10)
ax.legend(fontsize=11, frameon=False, loc="lower right")
ax.set_xlim(0, pivot["total"].max() * 1.15)

plt.tight_layout()
plt.savefig("/mnt/results/cell_type_qc/total_cellcount_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/total_cellcount_per_tissue.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


Done.


Check testis mouse data in detail

In [33]:

import pandas as pd

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]

testis = df[df['tissue'] == 'testis']

print("=== Mice in testis ===")
print(testis[['mouse', 'age']].drop_duplicates().sort_values('mouse'))

print(f"\nTotal mice: {testis['mouse'].nunique()}")

print("\n=== All mice in dataset ===")
all_mice = df[['mouse', 'age']].drop_duplicates().sort_values('mouse')
print(all_mice)

print("\n=== Mice NOT in testis ===")
all_mice_set = set(df['mouse'].unique())
testis_mice_set = set(testis['mouse'].unique())
missing = all_mice_set - testis_mice_set
print(sorted(missing))
print(f"Total missing: {len(missing)}")


=== Mice in testis ===
        mouse     age
759  mouse123  age27M
760  mouse124  age27M
761  mouse141   age3M
762  mouse142   age3M

Total mice: 4

=== All mice in dataset ===
        mouse     age
220  mouse122  age27M
221  mouse123  age27M
44   mouse124  age27M
45   mouse125  age27M
587  mouse126  age27M
588  mouse127  age27M
0    mouse128  age27M
483  mouse129  age27M
1    mouse130  age27M
333  mouse131  age27M
222  mouse135   age3M
223  mouse136   age3M
46   mouse137   age3M
122  mouse138   age3M
2    mouse139   age3M
485  mouse140   age3M
335  mouse141   age3M
718  mouse142   age3M
3    mouse143   age3M

=== Mice NOT in testis ===
['mouse122', 'mouse125', 'mouse126', 'mouse127', 'mouse128', 'mouse129', 'mouse130', 'mouse131', 'mouse135', 'mouse136', 'mouse137', 'mouse138', 'mouse139', 'mouse140', 'mouse143']
Total missing: 15


6x2 subplots: per tissue, horizontal bar by cell type (age3M + age27M stacked)

In [35]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

# Sum cell counts per tissue x cell_type x age
depth = (
    df.groupby(["tissue", "cell_type", "age"])["cell_count"]
    .sum()
    .reset_index()
)

sns.set_theme(style="ticks", font_scale=1.0)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = depth[depth["tissue"] == tissue].copy()

    pivot = sub.pivot(index="cell_type", columns="age", values="cell_count").fillna(0)
    # Ensure both age columns exist
    for col in ["age3M", "age27M"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["total"] = pivot["age3M"] + pivot["age27M"]
    pivot = pivot.sort_values("total", ascending=True)

    y = np.arange(len(pivot))
    bar_h = 0.6

    ax.barh(y, pivot["age3M"],  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.3)
    ax.barh(y, pivot["age27M"], height=bar_h, left=pivot["age3M"], color="#D6604D", label="age27M", edgecolor="white", linewidth=0.3)

    # Total label
    max_val = pivot["total"].max()
    for i, total in enumerate(pivot["total"]):
        ax.text(total + max_val * 0.02, i, f"{int(total/1000)}k" if total >= 1000 else str(int(total)),
                va="center", ha="left", fontsize=8)

    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis="x", labelsize=9)
    ax.set_xlim(0, max_val * 1.2)

# x-axis label on bottom panels only
for col in range(n_cols):
    axes[1, col].set_xlabel("Cell count", fontsize=11)

# Hide unused
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

# Shared legend
from matplotlib.patches import Patch
legend_handles = [Patch(color="#4393C3", label="age3M"), Patch(color="#D6604D", label="age27M")]
fig.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(0.98, 0.02),
           fontsize=12, frameon=False)

fig.suptitle("Cell counts by cell type per tissue (age3M + age27M)", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/cellcount_by_celltype_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/cellcount_by_celltype_per_tissue.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


Done.


Add 10k pink vertical reference line to each subplot

In [37]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

depth = (
    df.groupby(["tissue", "cell_type", "age"])["cell_count"]
    .sum()
    .reset_index()
)

sns.set_theme(style="ticks", font_scale=1.0)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = depth[depth["tissue"] == tissue].copy()

    pivot = sub.pivot(index="cell_type", columns="age", values="cell_count").fillna(0)
    for col in ["age3M", "age27M"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["total"] = pivot["age3M"] + pivot["age27M"]
    pivot = pivot.sort_values("total", ascending=True)

    y = np.arange(len(pivot))
    bar_h = 0.6

    ax.barh(y, pivot["age3M"],  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.3)
    ax.barh(y, pivot["age27M"], height=bar_h, left=pivot["age3M"], color="#D6604D", label="age27M", edgecolor="white", linewidth=0.3)

    # 10k reference line
    ax.axvline(x=10000, color="#FF69B4", linewidth=1.5, linestyle="--", zorder=5)

    # Total label
    max_val = pivot["total"].max()
    for i, total in enumerate(pivot["total"]):
        ax.text(total + max_val * 0.02, i, f"{int(total/1000)}k" if total >= 1000 else str(int(total)),
                va="center", ha="left", fontsize=8)

    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis="x", labelsize=9)
    ax.set_xlim(0, max_val * 1.2)

for col in range(n_cols):
    axes[1, col].set_xlabel("Cell count", fontsize=11)

for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color="#4393C3", label="age3M"),
    mpatches.Patch(color="#D6604D", label="age27M"),
    plt.Line2D([0], [0], color="#FF69B4", linewidth=1.5, linestyle="--", label="10k threshold"),
]
fig.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(0.98, 0.02),
           fontsize=12, frameon=False)

fig.suptitle("Cell counts by cell type per tissue (age3M + age27M)", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/cellcount_by_celltype_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/cellcount_by_celltype_per_tissue.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


Done.


6x2 subplots: per tissue, horizontal bar by cell type total reads (age3M + age27M stacked) with reference line

In [39]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

depth = (
    df.groupby(["tissue", "cell_type", "age"])["total_reads"]
    .sum()
    .reset_index()
)

sns.set_theme(style="ticks", font_scale=1.0)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = depth[depth["tissue"] == tissue].copy()

    pivot = sub.pivot(index="cell_type", columns="age", values="total_reads").fillna(0)
    for col in ["age3M", "age27M"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["total"] = pivot["age3M"] + pivot["age27M"]
    pivot = pivot.sort_values("total", ascending=True)

    y = np.arange(len(pivot))
    bar_h = 0.6

    ax.barh(y, pivot["age3M"],  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.3)
    ax.barh(y, pivot["age27M"], height=bar_h, left=pivot["age3M"], color="#D6604D", label="age27M", edgecolor="white", linewidth=0.3)

    # 10M reference line
    ax.axvline(x=10e6, color="#FF69B4", linewidth=1.5, linestyle="--", zorder=5)

    # Total label
    max_val = pivot["total"].max()
    for i, total in enumerate(pivot["total"]):
        ax.text(total + max_val * 0.02, i, f"{total/1e6:.0f}M" if total >= 1e6 else f"{total/1000:.0f}k",
                va="center", ha="left", fontsize=8)

    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{x/1000:.0f}k"))
    ax.tick_params(axis="x", labelsize=9)
    ax.set_xlim(0, max_val * 1.2)

for col in range(n_cols):
    axes[1, col].set_xlabel("Total reads", fontsize=11)

for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color="#4393C3", label="age3M"),
    mpatches.Patch(color="#D6604D", label="age27M"),
    plt.Line2D([0], [0], color="#FF69B4", linewidth=1.5, linestyle="--", label="10M threshold"),
]
fig.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(0.98, 0.02),
           fontsize=12, frameon=False)

fig.suptitle("Total reads by cell type per tissue (age3M + age27M)", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/totalreads_by_celltype_per_tissue.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/totalreads_by_celltype_per_tissue.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


Done.


Verify RNA_reads_per_cell == total_reads / cell_count

In [41]:

import pandas as pd
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

df['computed'] = df['total_reads'] / df['cell_count']
df['diff'] = (df['RNA_reads_per_cell'] - df['computed']).abs()

print(f"Max absolute diff: {df['diff'].max():.6f}")
print(f"Mean absolute diff: {df['diff'].mean():.6f}")
print(f"Any diff > 0.01: {(df['diff'] > 0.01).sum()} rows")


Max absolute diff: 0.000000
Mean absolute diff: 0.000000
Any diff > 0.01: 0 rows


RNA_reads_per_cell QC: tissue level (2x6) and cell type level (2x6 per tissue)

In [43]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

df_meta = df[['mouse', 'age']].drop_duplicates()

n_3M  = df_meta[df_meta["age"] == "age3M"]["mouse"].nunique()
n_27M = df_meta[df_meta["age"] == "age27M"]["mouse"].nunique()

color_map = {
    "age3M":  sns.color_palette("Blues",  n_colors=n_3M  + 2)[2:],
    "age27M": sns.color_palette("Reds",   n_colors=n_27M + 2)[2:],
}
mouse_colors = {}
for age, cmap in color_map.items():
    mice = sorted(df_meta[df_meta["age"] == age]["mouse"].unique())
    for i, m in enumerate(mice):
        mouse_colors[m] = cmap[i]

mice_3M  = sorted(df_meta[df_meta["age"] == "age3M"]["mouse"].unique())
mice_27M = sorted(df_meta[df_meta["age"] == "age27M"]["mouse"].unique())
ordered_mice = mice_3M + mice_27M
tissue_order = sorted(df['tissue'].unique())

def make_mouse_legend():
    handles = []
    handles.append(mpatches.Patch(color="none", label="age3M"))
    for m in mice_3M:
        handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
    handles.append(mpatches.Patch(color="none", label=""))
    handles.append(mpatches.Patch(color="none", label="age27M"))
    for m in mice_27M:
        handles.append(mpatches.Patch(color=mouse_colors[m], label=m))
    return handles

sns.set_theme(style="ticks", font_scale=1.0)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. Tissue level: weighted mean RNA_reads_per_cell per tissue per mouse
#    = total_reads_tissue / total_cells_tissue
# ═══════════════════════════════════════════════════════════════════════════════
tissue_agg = (
    df.groupby(["tissue", "mouse"])
    .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
    .reset_index(name="rpc")
)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 6.0 * n_rows), sharey=False)
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = tissue_agg[tissue_agg["tissue"] == tissue]
    sub_mice = [m for m in ordered_mice if m in sub["mouse"].values]
    sub = sub.set_index("mouse").reindex(sub_mice).reset_index()

    colors = [mouse_colors[m] for m in sub["mouse"]]
    ax.bar(range(len(sub)), sub["rpc"], color=colors, width=0.7, edgecolor="white", linewidth=0.5)

    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=6)
    ax.set_xticks([])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=11)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))

for row in range(n_rows):
    axes[row, 0].set_ylabel("RNA reads per cell", fontsize=13)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.legend(handles=make_mouse_legend(), loc="center right", bbox_to_anchor=(1.07, 0.5),
           fontsize=11, frameon=False, handlelength=1.4, handleheight=1.2)
fig.suptitle("RNA reads per cell — tissue level", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/mnt/results/cell_type_qc/rpc_tissue_level.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/rpc_tissue_level.png", dpi=150, bbox_inches="tight")
plt.close()
print("1/2 done")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. Cell type level: horizontal bar per tissue, X = weighted mean RPC
#    aggregated across mice (age3M + age27M stacked)
# ═══════════════════════════════════════════════════════════════════════════════
ct_agg = (
    df.groupby(["tissue", "cell_type", "age"])
    .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
    .reset_index(name="rpc")
)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = ct_agg[ct_agg["tissue"] == tissue]

    pivot = sub.pivot(index="cell_type", columns="age", values="rpc").fillna(0)
    for col in ["age3M", "age27M"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["mean_rpc"] = (pivot["age3M"] + pivot["age27M"]) / ((pivot["age3M"] > 0).astype(int) + (pivot["age27M"] > 0).astype(int))
    pivot = pivot.sort_values("mean_rpc", ascending=True)

    y = np.arange(len(pivot))
    bar_h = 0.6

    ax.barh(y, pivot["age3M"],  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.3)
    ax.barh(y, pivot["age27M"], height=bar_h, color="#D6604D", label="age27M", edgecolor="white", linewidth=0.3, alpha=0.85)

    # mean RPC label
    max_val = max(pivot[["age3M", "age27M"]].max().max(), 1)
    for i, (r3, r27) in enumerate(zip(pivot["age3M"], pivot["age27M"])):
        mean_val = pivot["mean_rpc"].iloc[i]
        ax.text(max(r3, r27) + max_val * 0.02, i,
                f"{mean_val/1000:.1f}k" if mean_val >= 1000 else f"{mean_val:.0f}",
                va="center", ha="left", fontsize=7.5)

    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis="x", labelsize=9)
    ax.set_xlim(0, max_val * 1.25)

for col in range(n_cols):
    axes[1, col].set_xlabel("RNA reads per cell", fontsize=11)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color="#4393C3", label="age3M"),
    mpatches.Patch(color="#D6604D", label="age27M"),
]
fig.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(0.98, 0.02),
           fontsize=12, frameon=False)
fig.suptitle("RNA reads per cell — cell type level (age3M vs age27M)", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)
plt.savefig("/mnt/results/cell_type_qc/rpc_celltype_level.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/rpc_celltype_level.png", dpi=150, bbox_inches="tight")
plt.close()
print("2/2 done")


/tmp/ipykernel_331/1191585666.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
1/2 done
/tmp/ipykernel_331/1191585666.py:94: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
2/2 done


Fix cell type RPC plot — offset bars so age3M and age27M don't overlap

In [45]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

ct_agg = (
    df.groupby(["tissue", "cell_type", "age"], group_keys=False)
    .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
    .reset_index(name="rpc")
)

sns.set_theme(style="ticks", font_scale=1.0)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

bar_h = 0.35  # each bar height
gap   = 0.05  # gap between the two bars

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = ct_agg[ct_agg["tissue"] == tissue]

    pivot = sub.pivot(index="cell_type", columns="age", values="rpc").fillna(0)
    for col in ["age3M", "age27M"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["mean_rpc"] = (pivot["age3M"] + pivot["age27M"]) / (
        (pivot["age3M"] > 0).astype(int) + (pivot["age27M"] > 0).astype(int)
    )
    pivot = pivot.sort_values("mean_rpc", ascending=True)

    n_ct = len(pivot)
    # Centre positions for each cell type
    y = np.arange(n_ct)
    # age3M: upper bar, age27M: lower bar
    y_3M  = y + gap/2 + bar_h/2
    y_27M = y - gap/2 - bar_h/2

    ax.barh(y_3M,  pivot["age3M"],  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.3)
    ax.barh(y_27M, pivot["age27M"], height=bar_h, color="#D6604D", label="age27M", edgecolor="white", linewidth=0.3)

    # Label at end of longer bar
    max_val = max(pivot[["age3M", "age27M"]].max().max(), 1)
    for i, (r3, r27) in enumerate(zip(pivot["age3M"], pivot["age27M"])):
        longer = max(r3, r27)
        label = f"{longer/1000:.1f}k" if longer >= 1000 else f"{longer:.0f}"
        ax.text(longer + max_val * 0.02, y[i], label,
                va="center", ha="left", fontsize=7.5)

    # Y ticks at cell type centres
    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_ylim(-0.7, n_ct - 0.3)

    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis="x", labelsize=9)
    ax.set_xlim(0, max_val * 1.25)

for col in range(n_cols):
    axes[1, col].set_xlabel("RNA reads per cell", fontsize=11)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color="#4393C3", label="age3M"),
    mpatches.Patch(color="#D6604D", label="age27M"),
]
fig.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(0.98, 0.02),
           fontsize=12, frameon=False)
fig.suptitle("RNA reads per cell — cell type level (age3M vs age27M)", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/rpc_celltype_level.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/rpc_celltype_level.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


/tmp/ipykernel_352/2099835417.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
Done.


RPC cell type level — label every bar + 1500 reference line

In [47]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/cell-counts.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

ct_agg = (
    df.groupby(["tissue", "cell_type", "age"], group_keys=False)
    .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
    .reset_index(name="rpc")
)

sns.set_theme(style="ticks", font_scale=1.0)

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

bar_h = 0.35
gap   = 0.05

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = ct_agg[ct_agg["tissue"] == tissue]

    pivot = sub.pivot(index="cell_type", columns="age", values="rpc").fillna(0)
    for col in ["age3M", "age27M"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["mean_rpc"] = (pivot["age3M"] + pivot["age27M"]) / (
        (pivot["age3M"] > 0).astype(int) + (pivot["age27M"] > 0).astype(int)
    )
    pivot = pivot.sort_values("mean_rpc", ascending=True)

    n_ct = len(pivot)
    y     = np.arange(n_ct)
    y_3M  = y + gap/2 + bar_h/2
    y_27M = y - gap/2 - bar_h/2

    ax.barh(y_3M,  pivot["age3M"],  height=bar_h, color="#4393C3", label="age3M",  edgecolor="white", linewidth=0.3)
    ax.barh(y_27M, pivot["age27M"], height=bar_h, color="#D6604D", label="age27M", edgecolor="white", linewidth=0.3)

    # Label every bar
    max_val = max(pivot[["age3M", "age27M"]].max().max(), 1)
    for i, (r3, r27) in enumerate(zip(pivot["age3M"], pivot["age27M"])):
        for val, ypos in [(r3, y_3M[i]), (r27, y_27M[i])]:
            if val > 0:
                label = f"{val/1000:.1f}k" if val >= 1000 else f"{int(val)}"
                ax.text(val + max_val * 0.015, ypos, label,
                        va="center", ha="left", fontsize=7)

    # 1500 reference line
    ax.axvline(x=1500, color="#FF69B4", linewidth=1.5, linestyle="--", zorder=5)

    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_ylim(-0.7, n_ct - 0.3)
    ax.set_title(tissue, fontsize=13, fontweight="bold", pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x))))
    ax.tick_params(axis="x", labelsize=9)
    ax.set_xlim(0, max_val * 1.25)

for col in range(n_cols):
    axes[1, col].set_xlabel("RNA reads per cell", fontsize=11)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color="#4393C3", label="age3M"),
    mpatches.Patch(color="#D6604D", label="age27M"),
    plt.Line2D([0], [0], color="#FF69B4", linewidth=1.5, linestyle="--", label="1500 threshold"),
]
fig.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(0.98, 0.02),
           fontsize=12, frameon=False)
fig.suptitle("RNA reads per cell — cell type level (age3M vs age27M)", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/cell_type_qc/rpc_celltype_level.svg", bbox_inches="tight")
plt.savefig("/mnt/results/cell_type_qc/rpc_celltype_level.png", dpi=150, bbox_inches="tight")
plt.close()
print("Done.")


/tmp/ipykernel_370/1309419476.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
Done.


Run consolidated QC script end-to-end

In [49]:

cd /mnt/results && python cell_counts_qc.py


/mnt/results/cell_counts_qc.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
/mnt/results/cell_counts_qc.py:338: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["total_reads"].sum() / x["cell_count"].sum())
All figures saved to /mnt/results/cell_type_qc


Check data structure: tissues, histone marks, columns

In [51]:

import pandas as pd

df = pd.read_csv("/mnt/user-uploads/df_dge_count.csv", index_col=0)
df['tissue'] = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

print(f"Shape: {df.shape}")
print(f"\nHistone marks: {df['histone_mark'].unique()}")
print(f"\nTissues: {sorted(df['tissue'].unique())}")
print(f"\nRows per tissue:\n{df.groupby('tissue').size()}")


Shape: (193, 10)

Histone marks: ['H3K9me3']

Tissues: ['BAT', 'brainCB', 'brainFC', 'brainHip', 'colon', 'heart', 'kidney', 'liver', 'lung', 'muscle', 'spleen', 'testis']

Rows per tissue:
tissue
BAT         11
brainCB     19
brainFC     25
brainHip    28
colon       18
heart       11
kidney      18
liver        8
lung        16
muscle      16
spleen      11
testis      12
dtype: int64


Plot diff bins per tissue: horizontal grouped bar (up_pval / down_pval) per cell type

In [53]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

df = pd.read_csv("/mnt/user-uploads/df_dge_count.csv", index_col=0)
df['tissue']    = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

UP_COLOR   = "#E8735A"   # warm red — up
DOWN_COLOR = "#5B8DB8"   # steel blue — down

sns.set_theme(style="ticks", font_scale=1.0)

bar_h = 0.35
gap   = 0.05

# Adaptive layout: 2 rows x 6 cols
n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()

    # Sort by total_pval descending so largest bars at top
    sub = sub.sort_values("total_pval", ascending=True)
    cell_types = sub['cell_type'].tolist()
    n_ct = len(cell_types)

    y     = np.arange(n_ct)
    y_up  = y + gap / 2 + bar_h / 2   # up bar: above centre
    y_dn  = y - gap / 2 - bar_h / 2   # down bar: below centre

    ax.barh(y_up, sub['up_pval'],   height=bar_h, color=UP_COLOR,   edgecolor="white", linewidth=0.3)
    ax.barh(y_dn, sub['down_pval'], height=bar_h, color=DOWN_COLOR, edgecolor="white", linewidth=0.3)

    # Labels on each bar
    max_val = max(sub[['up_pval', 'down_pval']].max().max(), 1)
    for i, (u, d) in enumerate(zip(sub['up_pval'], sub['down_pval'])):
        ax.text(u + max_val * 0.015, y_up[i], str(int(u)), va='center', ha='left', fontsize=7)
        ax.text(d + max_val * 0.015, y_dn[i], str(int(d)), va='center', ha='left', fontsize=7)

    ax.set_yticks(y)
    ax.set_yticklabels(cell_types, fontsize=9)
    ax.set_ylim(-0.7, n_ct - 0.3)
    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis='x', labelsize=9)
    ax.set_xlim(0, max_val * 1.25)

for col in range(n_cols):
    axes[1, col].set_xlabel("Number of diff bins (p-val)", fontsize=11)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color=UP_COLOR,   label="up_pval"),
    mpatches.Patch(color=DOWN_COLOR, label="down_pval"),
]
fig.legend(handles=legend_handles, loc="lower right",
           bbox_to_anchor=(0.98, 0.02), fontsize=12, frameon=False)

fig.suptitle("Differential bins per cell type (H3K9me3, p-val threshold)",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

os.makedirs("/mnt/results/dge", exist_ok=True)
plt.savefig("/mnt/results/dge/diff_bins_pval_per_tissue.svg", bbox_inches='tight')
plt.savefig("/mnt/results/dge/diff_bins_pval_per_tissue.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done.")


Done.


Move legend to bottom center of figure

In [55]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/df_dge_count.csv", index_col=0)
df['tissue']    = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

UP_COLOR   = "#E8735A"
DOWN_COLOR = "#5B8DB8"

sns.set_theme(style="ticks", font_scale=1.0)

bar_h = 0.35
gap   = 0.05

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy().sort_values("total_pval", ascending=True)
    cell_types = sub['cell_type'].tolist()
    n_ct = len(cell_types)

    y     = np.arange(n_ct)
    y_up  = y + gap / 2 + bar_h / 2
    y_dn  = y - gap / 2 - bar_h / 2

    ax.barh(y_up, sub['up_pval'],   height=bar_h, color=UP_COLOR,   edgecolor="white", linewidth=0.3)
    ax.barh(y_dn, sub['down_pval'], height=bar_h, color=DOWN_COLOR, edgecolor="white", linewidth=0.3)

    max_val = max(sub[['up_pval', 'down_pval']].max().max(), 1)
    for i, (u, d) in enumerate(zip(sub['up_pval'], sub['down_pval'])):
        ax.text(u + max_val * 0.015, y_up[i], str(int(u)), va='center', ha='left', fontsize=7)
        ax.text(d + max_val * 0.015, y_dn[i], str(int(d)), va='center', ha='left', fontsize=7)

    ax.set_yticks(y)
    ax.set_yticklabels(cell_types, fontsize=9)
    ax.set_ylim(-0.7, n_ct - 0.3)
    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis='x', labelsize=9)
    ax.set_xlim(0, max_val * 1.25)

for col in range(n_cols):
    axes[1, col].set_xlabel("Number of diff bins (p-val)", fontsize=11)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color=UP_COLOR,   label="up_pval"),
    mpatches.Patch(color=DOWN_COLOR, label="down_pval"),
]
fig.legend(handles=legend_handles,
           loc="lower center", bbox_to_anchor=(0.5, -0.02),
           ncol=2, fontsize=12, frameon=False)

fig.suptitle("Differential bins per cell type (H3K9me3, p-val threshold)",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/dge/diff_bins_pval_per_tissue.svg", bbox_inches='tight')
plt.savefig("/mnt/results/dge/diff_bins_pval_per_tissue.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done.")


Done.


Plot diff bins per tissue using FDR threshold (up_fdr / down_fdr)

In [57]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

df = pd.read_csv("/mnt/user-uploads/df_dge_count.csv", index_col=0)
df['tissue']    = df['tissue_ct'].str.split('-').str[0]
df['cell_type'] = df['tissue_ct'].str.split('-', n=1).str[1]

tissue_order = sorted(df['tissue'].unique())

UP_COLOR   = "#E8735A"
DOWN_COLOR = "#5B8DB8"

sns.set_theme(style="ticks", font_scale=1.0)

bar_h = 0.35
gap   = 0.05

n_cols, n_rows = 6, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 6.5 * n_rows))
axes_flat = axes.flatten()

for idx, tissue in enumerate(tissue_order):
    ax = axes_flat[idx]
    sub = df[df['tissue'] == tissue].copy()
    sub['total_fdr'] = sub['up_fdr'] + sub['down_fdr']
    sub = sub.sort_values("total_fdr", ascending=True)
    cell_types = sub['cell_type'].tolist()
    n_ct = len(cell_types)

    y     = np.arange(n_ct)
    y_up  = y + gap / 2 + bar_h / 2
    y_dn  = y - gap / 2 - bar_h / 2

    ax.barh(y_up, sub['up_fdr'],   height=bar_h, color=UP_COLOR,   edgecolor="white", linewidth=0.3)
    ax.barh(y_dn, sub['down_fdr'], height=bar_h, color=DOWN_COLOR, edgecolor="white", linewidth=0.3)

    max_val = max(sub[['up_fdr', 'down_fdr']].max().max(), 1)
    for i, (u, d) in enumerate(zip(sub['up_fdr'], sub['down_fdr'])):
        ax.text(u + max_val * 0.015, y_up[i], str(int(u)), va='center', ha='left', fontsize=7)
        ax.text(d + max_val * 0.015, y_dn[i], str(int(d)), va='center', ha='left', fontsize=7)

    ax.set_yticks(y)
    ax.set_yticklabels(cell_types, fontsize=9)
    ax.set_ylim(-0.7, n_ct - 0.3)
    ax.set_title(tissue, fontsize=13, fontweight='bold', pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis='x', labelsize=9)
    ax.set_xlim(0, max_val * 1.25)

for col in range(n_cols):
    axes[1, col].set_xlabel("Number of diff bins (FDR)", fontsize=11)
for idx in range(len(tissue_order), len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_handles = [
    mpatches.Patch(color=UP_COLOR,   label="up_fdr"),
    mpatches.Patch(color=DOWN_COLOR, label="down_fdr"),
]
fig.legend(handles=legend_handles,
           loc="lower center", bbox_to_anchor=(0.5, -0.02),
           ncol=2, fontsize=12, frameon=False)

fig.suptitle("Differential bins per cell type (H3K9me3, FDR threshold)",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=4, w_pad=3)

plt.savefig("/mnt/results/dge/diff_bins_fdr_per_tissue.svg", bbox_inches='tight')
plt.savefig("/mnt/results/dge/diff_bins_fdr_per_tissue.png", dpi=150, bbox_inches='tight')
plt.close()
print("Done.")


Done.


Run diff bins plot script to verify

In [59]:
python /mnt/results/dge/diff_bins_plot.py

Saved: diff_bins_pval_per_tissue
Saved: diff_bins_fdr_per_tissue
